In [1]:
# Cell 1: 导入库
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

In [2]:
# Cell 2: 准备文档（三句话，每句换行）
with open("test_rag.txt", "w") as f:
    f.write("PyTorch是一个深度学习框架。\nONNX用于模型转换。\nTensorRT用于推理加速。\n")

In [3]:
# Cell 3: 加载并分块
loader = TextLoader("test_rag.txt")
documents = loader.load()

text_splitter = CharacterTextSplitter(
    chunk_size=20, 
    chunk_overlap=0,
    separator="\n",
    keep_separator=False  # 去除换行字符\n
)
docs = text_splitter.split_documents(documents)

print(f"分成了 {len(docs)} 个 chunk")
print(docs[0])
all_texts = [doc.page_content for doc in docs]
print(all_texts)

分成了 3 个 chunk
page_content='PyTorch是一个深度学习框架。' metadata={'source': 'test_rag.txt'}
['PyTorch是一个深度学习框架。', 'ONNX用于模型转换。', 'TensorRT用于推理加速。']


In [4]:
# Cell 4: 向量化
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(docs, embeddings)

print(len(vectorstore._collection.get()['documents']))

3


In [5]:
# Cell 5: 检索并控制返回数量
query = "深度学习框架"

# 返回1个最相关的chunk
results_1 = vectorstore.similarity_search(query, k=1)
print("k=1 的结果:", results_1[0].page_content)

# 返回2个最相关的chunk
results_2 = vectorstore.similarity_search(query, k=2)
print("k=2 的结果:")
for i, r in enumerate(results_2):
    print(f"  {i+1}: {r.page_content}")

# 返回3个最相关的chunk
results_3 = vectorstore.similarity_search(query, k=3)
print("k=3 的结果:")
for i, r in enumerate(results_3):
    print(f"  {i+1}: {r.page_content}")

k=1 的结果: PyTorch是一个深度学习框架。
k=2 的结果:
  1: PyTorch是一个深度学习框架。
  2: ONNX用于模型转换。
k=3 的结果:
  1: PyTorch是一个深度学习框架。
  2: ONNX用于模型转换。
  3: TensorRT用于推理加速。


In [6]:
# Cell 6: 测试不同query
queries = ["PyTorch", "ONNX", "TensorRT", "深度学习", "模型转换"]

for q in queries:
    results = vectorstore.similarity_search(q, k=1)
    print(f"'{q}' → {results[0].page_content}")

'PyTorch' → PyTorch是一个深度学习框架。
'ONNX' → ONNX用于模型转换。
'TensorRT' → TensorRT用于推理加速。
'深度学习' → PyTorch是一个深度学习框架。
'模型转换' → ONNX用于模型转换。


### k=1 和 k=3 有什么区别？
- k=1的时候会返回最相近的1个结果，k=3的时候会返回最相近的3个结果。因为一共就只有3个chunk/vector，所以k=3的时候会返回全部chunk/vector。
### 为什么 "深度学习框架" 会匹配到 "PyTorch" 而不是 "ONNX"？
- 和“深度学习框架”最相近的vector是“PyTorch是一个深度学习框架。”。这大概率是因为“深度学习框架”在此vector中出现。
- 补充：语义相似度也起到了作用。
### 如果换一个query，返回结果会变吗？
- 不一定，取决于模型对于关键词的分析。比如说query“深度学习框架”和“PyTorch”会返回同样的结果，而query“模型转换”就会返回完全不一样的结果。这是针对于k=1和k=2的情况下。k=3的时候所有query都会返回同样的结果。